# 02. 빈자리 예측 모델 학습 / 평가

학습 코드는 `ml/jariml` 패키지에 있고, 이 노트북은 그걸 불러서 실행하고 결과를 본다.
실행 전 저장소를 최신으로 받는다 (`git pull`). 이 노트북은 `ml/notebooks/` 안에서 연다.

## 1. 설정

In [ ]:
import os, sys
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
sys.path.insert(0, os.path.abspath(".."))          # ml 폴더를 불러올 수 있게
import logging
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from jariml import load_config, Predictor
from jariml.train import run
from jariml.evaluate import by_lot

logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
plt.rcParams["font.family"] = next((f for f in ["Malgun Gothic", "AppleGothic", "NanumGothic"]
                                    if f in {x.name for x in fm.fontManager.ttflist}), "DejaVu Sans")
plt.rcParams["axes.unicode_minus"] = False
cfg = load_config()                                # ml/config.yaml

## 2. 학습 실행

원본 → 전처리 → 기간 분할 → 학습 → 평가까지 한 번에 돈다 (1~3분). `save=False`라 모델 파일은 저장하지 않는다.
운영 모델을 만들 때는 터미널에서 `python -m jariml.train`.

In [ ]:
out = run(cfg, save=False)

## 3. 기간 분할

In [ ]:
out["split"].describe()

## 4. 학습 곡선: 반복 횟수별 검증 오차

In [ ]:
curve = out["curve"] * 100
best = out["summary"]["best_iter"]
plt.figure(figsize=(8, 3.5))
plt.plot(range(1, len(curve) + 1), curve)
plt.axvline(best, color="tab:red", ls="--", label=f"선택: {best}회")
plt.xlabel("반복 횟수"); plt.ylabel("검증 점유율 MAE(%p)"); plt.legend(); plt.grid(alpha=0.3); plt.show()

## 5. 평가: 기준선과 비교 (평가 기간, 빈자리 대수)

- 지금값 유지: 도착할 때도 지금과 같다고 본다
- 지금+평소변화: 지금 값에 평소 곡선의 변화량만 더한다
- 모델: Gradient Boosting

In [ ]:
table = out["comparison"]
display(table.round(3))

mae = table["MAE(대)"].unstack()
mae.plot(marker="o", figsize=(8, 4), title="예측 거리별 빈자리 오차 (MAE, 대)")
plt.xlabel("예측 거리(분)"); plt.ylabel("MAE(대)"); plt.grid(alpha=0.3); plt.show()
print("1시간 뒤: 모델이 '지금값 유지'보다", round((1 - mae.loc[60, "모델"] / mae.loc[60, "지금값 유지"]) * 100, 1),
      "%, '지금+평소변화'보다", round((1 - mae.loc[60, "모델"] / mae.loc[60, "지금+평소변화"]) * 100, 1), "% 오차가 작다")

## 6. 도착 시각대별 오차 (1시간 뒤 예측)

In [ ]:
hourly = out["hourly"]
hourly["MAE(대)"].plot.bar(figsize=(10, 3.5), title="도착 시각별 1시간 뒤 빈자리 오차 (대)")
plt.xlabel("도착 시각(시)"); plt.ylabel("MAE(대)"); plt.show()
hourly.round(2).T

## 7. 주차장별 오차 (1시간 뒤 예측)

In [ ]:
lm = out["tables"].lot_master
lot_err = by_lot(out["test"], out["pred_test"]).rename("평가 MAE(대)").to_frame()
lot_err["주차장"] = lm.PKLT_NM.reindex(lot_err.index).str.replace(" 공영주차장(시)", "")
lot_err["총면"] = lm.TPKCT.reindex(lot_err.index).astype(int)
lot_err["오차/총면(%)"] = lot_err["평가 MAE(대)"] / lot_err.총면 * 100
lot_err["신뢰도"] = out["confidence"].label.reindex(lot_err.index)
lot_err.sort_values("평가 MAE(대)", ascending=False).round(2)

## 8. 확률 검증: '빈자리 3대 이상일 확률'이 실제와 맞는가

예측 확률 구간별로 실제로 자리가 있었던 비율을 비교한다. 두 값이 비슷할수록 믿을 수 있는 확률이다.

In [ ]:
print({k: round(v, 4) for k, v in out["summary"].items() if "Brier" in k or "비율" in k})
cal = out["calibration"]
display(cal.round(3))
plt.figure(figsize=(4.5, 4.5))
plt.plot([0, 1], [0, 1], "k--", lw=1, label="완벽한 확률")
plt.scatter(cal.예측확률평균, cal.실제비율, s=np.sqrt(cal.건수) * 3, label="구간 (원 크기 = 건수)")
plt.xlabel("예측 확률"); plt.ylabel("실제 자리 있었던 비율"); plt.legend(); plt.grid(alpha=0.3); plt.show()

## 9. 어떤 입력이 중요한가 (평가 기간 일부로 계산, 1~2분)

In [ ]:
from sklearn.inspection import permutation_importance
from jariml.features import FEATURES

te = out["test"].sample(30000, random_state=0)
model_eval = out["artifact"]["model"]          # 전체 기간 재학습 모델이라 평가 기간도 본 상태: 순서만 참고
imp = permutation_importance(model_eval, te[FEATURES], te.y - te.occ, n_repeats=3, random_state=0,
                             scoring="neg_mean_absolute_error")
pd.Series(imp.importances_mean * 100, index=FEATURES).sort_values().plot.barh(
    figsize=(7, 5), title="입력을 섞었을 때 늘어나는 오차 (%p)"); plt.show()

## 10. 예측 예시

기준 시각과 도착 시각을 바꿔 가며 볼 수 있다. 결과는 대수와 확률로 나온다.

In [ ]:
p = Predictor(out["artifact"], out["metrics"])
recent = p.recent_from_data_dir()
NOW = recent.t.max() - pd.Timedelta(hours=20)          # 기준 시각 (예: 데이터 끝에서 20시간 전)
ARRIVAL = NOW + pd.Timedelta(minutes=60)               # 도착 시각
r = p.predict(recent, ARRIVAL, now=NOW)
r.sort_values("prob_free_ge_3")[["PKLT_NM", "cap", "now_cnt", "now_free", "pred_cnt", "pred_free",
                                 "prob_free_ge_3", "confidence"]].head(15)

## 11. 자리 확보 가능한 마지막 도착 시각 (확률 80% 기준)

In [ ]:
d = p.deadline(recent, now=NOW, min_prob=0.8)
d[d.note != "2시간 내내 가능"]